# NumPy 3/3 : Agrégation et broadcasting

**Objectifs** : résumer une array en quelques valeurs (somme, moyenne, etc.) le long d'un axe choisi, puis comprendre comment NumPy combine des arrays de formes différentes (**broadcasting**).

*Durée estimée : 40 min.*

In [ ]:
# Cellule de préparation : à exécuter en premier (ne fait rien en local)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("intro-data-python"):
    !git clone -q https://github.com/cdrutinuspro/intro-data-python.git
    %cd intro-data-python/notebooks

In [ ]:
import numpy as np

## 1. Agrégations de base

Une *agrégation* résume plusieurs valeurs en une seule.

In [ ]:
a = np.array([4, 8, 15, 16, 23, 42])

print("somme      :", a.sum())
print("moyenne    :", a.mean())
print("médiane    :", np.median(a))
print("min / max  :", a.min(), a.max())
print("écart-type :", a.std())
print("argmin/max :", a.argmin(), a.argmax())   # *positions* du min et du max

`argmin` et `argmax` renvoient la **position** de l'extremum, pas sa valeur.

## 2. Agrégations sur une matrice : l'argument `axis`

Sur une matrice, on peut agréger tout, ou **le long d'un axe** :

- `axis=0` : on « écrase » les lignes → un résultat **par colonne**
- `axis=1` : on « écrase » les colonnes → un résultat **par ligne**

In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])

print(m.sum())          # tout
print(m.sum(axis=0))    # par colonne
print(m.sum(axis=1))    # par ligne

Pour retenir : l'axe indiqué est celui qui **disparaît** du `shape` (ici `(2, 3)` → `axis=0` donne `(3,)`, `axis=1` donne `(2,)`).

### Exemple concret : notes d'étudiants

Chaque ligne est un étudiant, chaque colonne une matière.

In [ ]:
notes = np.array([[12, 14,  9],
                  [ 8, 11, 13],
                  [15, 16, 17],
                  [ 6,  7, 10]])

print("Moyenne par étudiant :", notes.mean(axis=1))
print("Moyenne par matière  :", notes.mean(axis=0))
print("Meilleur étudiant (indice) :", notes.mean(axis=1).argmax())

### Exercices (agrégation)

### ✏️ Exercice 1

Avec la matrice `notes` ci-dessus, affichez la note maximale de chaque matière, puis la note minimale de chaque étudiant.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
print(notes.max(axis=0))
print(notes.min(axis=1))
```

</details>

### ✏️ Exercice 2

Combien d'étudiants ont une moyenne supérieure ou égale à 10 ? (indice : combinez agrégation et masque)

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
moyennes = notes.mean(axis=1)
print((moyennes >= 10).sum())
```

</details>

### ✏️ Exercice 3

Pour chaque matière, calculez la proportion d'étudiants ayant une note >= 10.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
print((notes >= 10).mean(axis=0))
```

</details>

## 3. Broadcasting

Que se passe-t-il quand on additionne deux arrays de **formes différentes** ? NumPy « étend » virtuellement la plus petite pour l'adapter, sans copier les données.

### Cas 1 : array et scalaire

In [ ]:
a = np.array([1, 2, 3])
print(a + 10)
print(a * 2)

### Cas 2 : matrice et vecteur ligne

Le vecteur est ajouté à **chaque ligne**.

In [ ]:
m = np.array([[ 0,  0,  0],
              [10, 10, 10],
              [20, 20, 20]])
v = np.array([1, 2, 3])

print(m.shape, v.shape)
print(m + v)

### Cas 3 : matrice et vecteur colonne

Pour ajouter une valeur **par ligne**, il faut une colonne, c'est-à-dire un `shape` `(3, 1)`. On l'obtient avec `reshape` ou `[:, np.newaxis]`.

In [ ]:
col = np.array([100, 200, 300]).reshape(3, 1)
print(col.shape)
print(m + col)

### Cas 4 : deux vecteurs → une table

Un vecteur colonne et un vecteur ligne donnent une matrice (par exemple une table de multiplication).

In [ ]:
lignes = np.arange(1, 6).reshape(5, 1)
cols = np.arange(1, 6)
print(lignes * cols)

### Les règles du broadcasting

NumPy compare les `shape` **de droite à gauche**. Deux dimensions sont compatibles si :

1. elles sont **égales**, ou
2. l'une d'elles vaut **1** (elle est alors étirée).

Sinon : erreur.

| A | B | Résultat |
|---|---|---|
| `(3, 3)` | `(3,)` | ✅ `(3, 3)` |
| `(3, 3)` | `(3, 1)` | ✅ `(3, 3)` |
| `(5, 1)` | `(5,)` | ✅ `(5, 5)` |
| `(3, 4)` | `(3,)` | ❌ erreur (4 ≠ 3) |

In [ ]:
a = np.ones((3, 4))
b = np.array([1, 2, 3])
try:
    a + b
except ValueError as e:
    print("Erreur :", e)

### Application : centrer-réduire des colonnes

Une opération classique en analyse de données : retrancher la moyenne et diviser par l'écart-type de **chaque colonne**.

In [ ]:
rng = np.random.default_rng(1)
X = rng.normal(loc=[10, 100, 1000], scale=[1, 10, 100], size=(200, 3))

mu = X.mean(axis=0)      # shape (3,)
sigma = X.std(axis=0)    # shape (3,)
Z = (X - mu) / sigma     # broadcasting : (200, 3) - (3,)

print("moyennes après :", Z.mean(axis=0).round(3))
print("écarts-types après :", Z.std(axis=0).round(3))

### Piège : `keepdims`

Pour centrer **par ligne** (et non par colonne), la moyenne doit être une colonne `(n, 1)`. L'option `keepdims=True` conserve la dimension agrégée :

In [ ]:
m = np.array([[1., 2., 3.],
              [4., 5., 6.]])
moy = m.mean(axis=1, keepdims=True)
print(moy.shape)
print(m - moy)

## Récapitulatif

- `sum`, `mean`, `min`, `max`, `std`, `argmin`, `argmax` : agrégations
- `axis=0` → un résultat par **colonne** ; `axis=1` → un résultat par **ligne**
- Broadcasting : les `shape` sont comparées de droite à gauche, dimensions égales ou égales à 1
- `keepdims=True` ou `reshape` / `np.newaxis` pour obtenir des colonnes

---
## Exercices (broadcasting)

### ✏️ Exercice 4

Soit `m = np.arange(12).reshape(3, 4)`. Ajoutez `[100, 200, 300, 400]` à chaque ligne.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
m = np.arange(12).reshape(3, 4)
print(m + np.array([100, 200, 300, 400]))
```

</details>

### ✏️ Exercice 5

Avec la même `m`, multipliez chaque **ligne** respectivement par 1, 10 et 100.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
print(m * np.array([1, 10, 100]).reshape(3, 1))
```

</details>

### ✏️ Exercice 6

Soit `prix = np.array([10.0, 20.0, 5.0])` (3 produits) et `quantites` une matrice 4×3 (4 magasins) : `quantites = np.array([[1,2,3],[0,1,0],[5,5,5],[2,0,1]])`. Calculez le chiffre d'affaires de chaque magasin.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
prix = np.array([10.0, 20.0, 5.0])
quantites = np.array([[1,2,3],[0,1,0],[5,5,5],[2,0,1]])
print((quantites * prix).sum(axis=1))
```

</details>

### ✏️ Exercice 7

**Défi** : sans boucle, construisez la matrice 6×6 `D` telle que `D[i, j] = |i - j|` (distance entre indices).

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
i = np.arange(6).reshape(6, 1)
j = np.arange(6)
D = np.abs(i - j)
print(D)
```

</details>